# Custom Trace Attributes: Tagging a Business Decision, Not Just a Technical One

Demo 02 showed the trace tree Strands gives you automatically. This notebook adds attributes that mean something to a **business**, not just to the SDK, onto that same tree — two ways: a static `trace_attributes` dict at agent creation, and a dynamic `AfterToolCallEvent` hook that tags a span only when a business rule fires at runtime.

## The Tools

Same travel agent as demos 01-02 — real APIs, no injected failures.

| Tool | What it does | Key behavior |
|------|-------------|--------------|
| `search_flights(origin, destination, departure_date)` | Searches real one-way fares via the Duffel **sandbox** API | Returns up to 5 offers sorted by price |
| `get_weather(city, target_date)` | Real daily forecast via Open-Meteo (no auth) | Only covers dates within ~16 days from today |
| `book_flight(offer_id, given_name, family_name, amount, currency)` | Writes a confirmed booking to a local SQLite ledger | No paid order is ever placed |


## Docs this notebook is built from

- [Custom Attribute Tracking](https://strandsagents.com/docs/user-guide/observability-evaluation/traces/#82-custom-attribute-tracking) — static `trace_attributes` at agent creation
- [Custom Spans](https://strandsagents.com/docs/user-guide/observability-evaluation/traces/#83-custom-spans) — reaching the currently active span from your own code
- [Hooks](https://strandsagents.com/docs/user-guide/concepts/agents/hooks/) — `HookProvider`, `HookRegistry`, `AfterToolCallEvent`


## Setup

Two new imports vs demo 02. From `strands.hooks`: `AfterToolCallEvent`, `HookProvider`, and `HookRegistry` — hooks are Strands' mechanism for reacting to events in the agent's lifecycle. And from `opentelemetry`: `trace` — the standard OTEL API, which we'll use to reach the currently active span.


In [1]:
import json
import os
from datetime import datetime, timedelta

from dotenv import load_dotenv
from opentelemetry import trace
from strands import Agent
from strands.hooks import AfterToolCallEvent, HookProvider, HookRegistry
from strands.models.openai import OpenAIModel  # OpenAI-compatible interface via Strands SDK
from strands.telemetry import StrandsTelemetry

import travel_tools as T

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY not set. Get yours at https://platform.openai.com/api-keys "
        "and add it to a .env file."
    )

strands_telemetry = StrandsTelemetry()
strands_telemetry.setup_console_exporter()

model = OpenAIModel(model_id="gpt-4o-mini")

SYSTEM_PROMPT = (
    "You are a travel assistant. Search flights, check the weather at the destination, "
    "and book the best option for the traveler without asking for confirmation. Be concise."
)

TRIP_DATE = (datetime.now() + timedelta(days=5)).strftime("%Y-%m-%d")
TRIP_PROMPT = (
    f"Book a one-way flight from JFK to MIA on {TRIP_DATE} for John Doe, "
    "and tell me if he'll need a jacket."
)

print("✅ Setup complete!")


✅ Setup complete!


## The business rule and the hook

`AfterToolCallEvent` fires right after every tool call finishes — while that tool's span is still the active OpenTelemetry span. So `trace.get_current_span()` inside the callback returns the REAL `execute_tool book_flight` span, and `span.set_attribute(...)` tags it in place.


In [2]:
VIP_THRESHOLD = 50.0  # USD; low on purpose so the Duffel sandbox fares actually cross it


class TagVipBookings(HookProvider):
    def __init__(self, threshold):
        self.threshold = threshold
        self.tagged = 0

    def register_hooks(self, registry: HookRegistry, **kwargs):
        registry.add_callback(AfterToolCallEvent, self._tag_if_vip)

    def _tag_if_vip(self, event: AfterToolCallEvent):
        if event.tool_use.get("name") != "book_flight":
            return
        amount = float(event.tool_use.get("input", {}).get("amount", 0))
        span = trace.get_current_span()
        is_vip = amount >= self.threshold
        span.set_attribute("business.booking_amount_usd", amount)
        span.set_attribute("business.vip_booking", is_vip)
        if is_vip:
            self.tagged += 1

print("✅ Hook defined")


✅ Hook defined


## The agent

Two additions vs the plain travel agent: `trace_attributes` (static — the session ID lands on every span) and `hooks=[vip_hook]` (dynamic — the business tag lands only on the span where the rule fires).


In [3]:
T.init_booking_db()
vip_hook = TagVipBookings(threshold=VIP_THRESHOLD)
agent = Agent(model=model, system_prompt=SYSTEM_PROMPT,
              tools=[T.search_flights, T.get_weather, T.book_flight],
              trace_attributes={"session.id": "demo-03-custom-trace-attributes"},
              hooks=[vip_hook])

result = agent(TRIP_PROMPT)



Tool #1: search_flights

Tool #2: get_weather
{
    "name": "chat",
    "context": {
        "trace_id": "0x4579a581a36c5642d6d967dd8270d7d2",
        "span_id": "0x23a4f1d0fc2c779a",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x99f6e470e4f14149",
    "start_time": "2026-07-23T03:43:57.158747Z",
    "end_time": "2026-07-23T03:43:59.922403Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-23T03:43:57.158748+00:00",
        "gen_ai.operation.name": "chat",
        "gen_ai.system": "strands-agents",
        "session.id": "demo-03-custom-trace-attributes",
        "gen_ai.request.model": "gpt-4o-mini",
        "gen_ai.event.end_time": "2026-07-23T03:43:59.922350+00:00",
        "gen_ai.usage.prompt_tokens": 498,
        "gen_ai.usage.input_tokens": 498,
        "gen_ai.usage.completion_tokens": 71,
        "gen_ai.usage.output_tokens": 71,
        "gen_ai.usage.total_tokens": 569

{
    "name": "execute_tool search_flights",
    "context": {
        "trace_id": "0x4579a581a36c5642d6d967dd8270d7d2",
        "span_id": "0x09e12edfe086fa0d",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x99f6e470e4f14149",
    "start_time": "2026-07-23T03:43:59.924209Z",
    "end_time": "2026-07-23T03:44:00.505428Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-23T03:43:59.924213+00:00",
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.system": "strands-agents",
        "gen_ai.tool.name": "search_flights",
        "gen_ai.tool.call.id": "call_J85V2P65LskcERw2qTi64zuR",
        "session.id": "demo-03-custom-trace-attributes",
        "gen_ai.tool.description": "Search one-way flight offers (Duffel sandbox). Present these to the traveler to choose from.\n\nReturns:\n    A dict with `offers`: up to 5 options, each with `offer_id`, `airline`,\n    `total_amoun

{
    "name": "execute_tool get_weather",
    "context": {
        "trace_id": "0x4579a581a36c5642d6d967dd8270d7d2",
        "span_id": "0xfb45df8e2698bf19",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x99f6e470e4f14149",
    "start_time": "2026-07-23T03:43:59.924846Z",
    "end_time": "2026-07-23T03:44:00.797183Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-23T03:43:59.924852+00:00",
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.system": "strands-agents",
        "gen_ai.tool.name": "get_weather",
        "gen_ai.tool.call.id": "call_dNke90YWS6sFcPRcfhhodVQv",
        "session.id": "demo-03-custom-trace-attributes",
        "gen_ai.tool.description": "Get the daily weather forecast for a city on a date (Open-Meteo), to advise on packing.\n\nReturns:\n    A dict with `city`, `temperature_max_c`, `temperature_min_c`, or an `error`.",
        "gen_ai.tool.

{
    "name": "execute_event_loop_cycle",
    "context": {
        "trace_id": "0x4579a581a36c5642d6d967dd8270d7d2",
        "span_id": "0x99f6e470e4f14149",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xf6c141341e28aa9b",
    "start_time": "2026-07-23T03:43:57.158539Z",
    "end_time": "2026-07-23T03:44:00.800416Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-23T03:43:57.158540+00:00",
        "gen_ai.operation.name": "execute_event_loop_cycle",
        "gen_ai.system": "strands-agents",
        "event_loop.cycle_id": "5d83d461-4f3b-4430-b373-9cb96d902270",
        "session.id": "demo-03-custom-trace-attributes",
        "gen_ai.event.end_time": "2026-07-23T03:44:00.800377+00:00"
    },
    "events": [
        {
            "name": "gen_ai.user.message",
            "timestamp": "2026-07-23T03:43:57.158565Z",
            "attributes": {
                "content": "[{\"text\


Tool #3: book_flight
{
    "name": "chat",
    "context": {
        "trace_id": "0x4579a581a36c5642d6d967dd8270d7d2",
        "span_id": "0x638e28546a382963",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x0ae4f48156d376db",
    "start_time": "2026-07-23T03:44:00.803213Z",
    "end_time": "2026-07-23T03:44:03.033818Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-23T03:44:00.803217+00:00",
        "gen_ai.operation.name": "chat",
        "gen_ai.system": "strands-agents",
        "session.id": "demo-03-custom-trace-attributes",
        "gen_ai.request.model": "gpt-4o-mini",
        "gen_ai.event.end_time": "2026-07-23T03:44:03.033767+00:00",
        "gen_ai.usage.prompt_tokens": 967,
        "gen_ai.usage.input_tokens": 967,
        "gen_ai.usage.completion_tokens": 52,
        "gen_ai.usage.output_tokens": 52,
        "gen_ai.usage.total_tokens": 1019,
        "gen_ai.server

...

 successfully booked with British Airways for $88.76. The booking reference is **BK-

4MLWI9**.

As for the weather in Miami, it will be warm, with a maximum temperature

 of 31.9°C and a minimum of 28.1°C. He will not need a jacket.{
    "name": "chat",
    "context": {
        "trace_id": "0x4579a581a36c5642d6d967dd8270d7d2",
        "span_id": "0x957d23d1b60b7562",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x8d3a7a2e0e75b9f6",
    "start_time": "2026-07-23T03:44:03.043055Z",
    "end_time": "2026-07-23T03:44:04.250703Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-23T03:44:03.043057+00:00",
        "gen_ai.operation.name": "chat",
        "gen_ai.system": "strands-agents",
        "session.id": "demo-03-custom-trace-attributes",
        "gen_ai.request.model": "gpt-4o-mini",
        "gen_ai.event.end_time": "2026-07-23T03:44:04.250653+00:00",
        "gen_ai.usage.prompt_tokens": 1068,
        "gen_ai.usage.input_tokens": 1068,
        "gen_ai.usage.completion_tokens": 86,
        "gen_ai.usage.output_tokens": 86,
        "gen_ai.usage.t

{
    "name": "execute_event_loop_cycle",
    "context": {
        "trace_id": "0x4579a581a36c5642d6d967dd8270d7d2",
        "span_id": "0x8d3a7a2e0e75b9f6",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xf6c141341e28aa9b",
    "start_time": "2026-07-23T03:44:03.042376Z",
    "end_time": "2026-07-23T03:44:04.252147Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-23T03:44:03.042390+00:00",
        "gen_ai.operation.name": "execute_event_loop_cycle",
        "gen_ai.system": "strands-agents",
        "event_loop.cycle_id": "e3fa06e5-7461-40ce-bd58-05f8233cdb57",
        "session.id": "demo-03-custom-trace-attributes",
        "event_loop.parent_cycle_id": "51b2ed40-89f7-436e-8ef2-f8e50a1b3252",
        "gen_ai.event.end_time": "2026-07-23T03:44:04.252128+00:00"
    },
    "events": [
        {
            "name": "gen_ai.user.message",
            "timestamp": "2026-07-23T03:44:

{
    "name": "invoke_agent Strands Agents",
    "context": {
        "trace_id": "0x4579a581a36c5642d6d967dd8270d7d2",
        "span_id": "0xf6c141341e28aa9b",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-07-23T03:43:57.158292Z",
    "end_time": "2026-07-23T03:44:04.253096Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "gen_ai.event.start_time": "2026-07-23T03:43:57.158295+00:00",
        "gen_ai.operation.name": "invoke_agent",
        "gen_ai.system": "strands-agents",
        "gen_ai.agent.name": "Strands Agents",
        "gen_ai.request.model": "gpt-4o-mini",
        "gen_ai.agent.tools": "[\"search_flights\", \"get_weather\", \"book_flight\"]",
        "session.id": "demo-03-custom-trace-attributes",
        "system_prompt": "You are a travel assistant. Search flights, check the weather at the destination, and book the best option for the traveler without asking for confirmati

## What just happened

Find the `execute_tool book_flight` span printed above and look at its attributes: the SDK's (`gen_ai.tool.name`, `gen_ai.tool.status`), the static `session.id`, and — added by our hook — `business.booking_amount_usd` and `business.vip_booking`. On the trace, not in the conversation: the model never sees them.


In [4]:
print(f"Bookings tagged business.vip_booking=True this run: {vip_hook.tagged}")
print()
print("Final response:")
print(result.message["content"][0]["text"])


Bookings tagged business.vip_booking=True this run: 1

Final response:
John Doe's one-way flight from JFK to MIA on July 27, 2026, has been successfully booked with British Airways for $88.76. The booking reference is **BK-4MLWI9**.

As for the weather in Miami, it will be warm, with a maximum temperature of 31.9°C and a minimum of 28.1°C. He will not need a jacket.


## Ground truth


In [5]:
print(json.dumps(T.query_booked_offers(), indent=2))


[
  {
    "booking_reference": "BK-4MLWI9",
    "offer_id": "off_0000B8cDqGP6LsIy4Mlwi9",
    "passenger": "John Doe",
    "amount": "88.76",
    "currency": "USD"
  }
]


## Key takeaways

- `trace_attributes` (static) tags EVERY span an agent produces — good for cross-cutting metadata like session or user id.
- An `AfterToolCallEvent` hook (dynamic) tags only the spans where a runtime condition is true — good for business facts that depend on the actual tool call's data.
- Both live in OpenTelemetry span attributes, entirely separate from the message list the model reads.

## References

- [Strands Agents: Traces — Custom Attribute Tracking](https://strandsagents.com/docs/user-guide/observability-evaluation/traces/#82-custom-attribute-tracking) · [Custom Spans](https://strandsagents.com/docs/user-guide/observability-evaluation/traces/#83-custom-spans)
- [Strands Agents: Hooks](https://strandsagents.com/docs/user-guide/concepts/agents/hooks/)
- Previous: [02 - OpenTelemetry Traces](../02-opentelemetry-traces/) · Next: [04 - AgentCore Observability](../04-agentcore-observability/)
